# LeetCode #1136 — Parallel Courses: Xếp lịch học song song

**Medium · Python 3 · Directed Graph, Topological Sort, BFS theo tầng**  
[Trang bài](https://leetcode.com/problems/parallel-courses/) · [Mô tả tham khảo (bài Premium)](https://leetcode.doocs.org/en/lc/1136/)

```python
class Solution:
    def minimumSemesters(self, n: int, relations: List[List[int]]) -> int:
        ...
```

**Mục tiêu của notebook:** hiểu *vì sao* mỗi lượt lấy tất cả môn không còn điều kiện tiên quyết sẽ cho **số học kỳ ít nhất**; biết nhận ra chu trình và không để môn mới được mở khóa học ngay trong cùng học kỳ.

> Bài #1136 có nhãn **Premium**; nội dung đề và constraints được đối chiếu qua nguồn tham khảo liên kết ở trên, không giả định có quyền đọc đề Premium trực tiếp trên LeetCode.

## 1. Sau bài này bạn sẽ làm được gì?

1. Phân biệt đồ thị **có hướng** và đồ thị vô hướng.
2. Tạo **adjacency list** (danh sách kề), tính **in-degree** (bậc vào) và dùng `deque`.
3. Giải cách mô phỏng từng học kỳ trước khi tối ưu.
4. Tự suy ra **Kahn's topological sort** (sắp xếp topo) theo từng lớp BFS.
5. Giải thích được tại sao chu trình trả `-1`, tại sao số lớp chính là đáp án tối ưu.
6. Tự chạy trace, test, rồi tự điền bài tập TODO.

## 2. Đề bài và các ví dụ

Có `n` môn học đánh số từ **1 đến n**. Cặp `[a,b]` trong `relations` nghĩa là: **phải học xong `a` ở học kỳ trước**, sau đó mới được học `b`. Trong một học kỳ có thể học bao nhiêu môn cũng được nếu đã hoàn thành mọi tiên quyết từ các học kỳ trước.

Tìm **số học kỳ ít nhất** để học hết; nếu không thể, trả `-1`.

| Input | Output | Giải thích |
|---|---:|---|
| `n=3, relations=[[1,3],[2,3]]` | `2` | Kỳ 1: `1,2`; kỳ 2: `3` |
| `n=3, relations=[[1,2],[2,3],[3,1]]` | `-1` | Ba môn khóa vòng lẫn nhau |
| `n=4, relations=[[1,3],[2,3],[3,4]]` | `3` | `1,2` → `3` → `4` |

**Ràng buộc tham khảo:** `1 <= n <= 5000`; `1 <= len(relations) <= 5000`; mỗi quan hệ có 2 số, từ 1 đến `n`, hai số khác nhau, các cặp không trùng. Ta vẫn thử `relations=[]` trong phần mở rộng để hiểu trường hợp không có điều kiện tiên quyết.

In [1]:
n_demo = 4
rel_demo = [[1, 3], [2, 3], [3, 4]]
print('Số môn:', n_demo)
print('Các quan hệ [môn trước, môn sau]:', rel_demo)
print('Lịch hợp lệ ngắn nhất: [1,2] -> [3] -> [4]')

Số môn: 4
Các quan hệ [môn trước, môn sau]: [[1, 3], [2, 3], [3, 4]]
Lịch hợp lệ ngắn nhất: [1,2] -> [3] -> [4]


## 3. Đọc một cạnh có hướng

Một cạnh `1 -> 3` nghĩa là muốn học **3** phải học xong **1** trước đó; **không** có nghĩa 3 là tiên quyết của 1. Hai môn không nối nhau có thể học song song. Đọc `relations` theo cách đó sẽ tránh việc xây nhầm chiều đồ thị.

**Nút (vertex):** một môn. **Cạnh (edge):** điều kiện tiên quyết.

In [2]:
for before, after in rel_demo:
    print(f'{before} -> {after}: học {before} trước {after}')

1 -> 3: học 1 trước 3
2 -> 3: học 2 trước 3
3 -> 4: học 3 trước 4


## 4. Python cơ bản: danh sách lồng nhau, `range`, `append`

Ta cần `graph[u]` lưu những môn được mở khóa **sau** khi học `u`.

- `range(n+1)` tạo index `0..n`, nhưng môn chỉ có nhãn `1..n`, nên index `0` không sử dụng.
- `[[] for _ in range(n+1)]` tạo **mỗi danh sách con độc lập**.
- `graph[u].append(v)` thêm môn `v` vào danh sách kề của `u`.

Không dùng `[[]] * (n+1)` vì mọi phần tử sẽ cùng trỏ tới *một* danh sách con.

In [3]:
graph_demo = [[] for _ in range(n_demo + 1)]
for u, v in rel_demo:
    graph_demo[u].append(v)
for course in range(1, n_demo + 1):
    print(f'graph[{course}] = {graph_demo[course]}')
assert graph_demo[3] == [4] and graph_demo[4] == []

wrong_graph = [[]] * 3
wrong_graph[1].append(99)
print('Sai nếu dùng [[]] * 3:', wrong_graph)
assert wrong_graph[0] == [99]

graph[1] = [3]
graph[2] = [3]
graph[3] = [4]
graph[4] = []
Sai nếu dùng [[]] * 3: [[99], [99], [99]]


## 5. In-degree là gì?

**Bậc vào `indegree[v]`** = số điều kiện tiên quyết của môn `v` **chưa được giải quyết**. Ban đầu, nó là số cạnh đi **vào** `v`.

Với `[[1,3],[2,3],[3,4]]`:

| Môn | 1 | 2 | 3 | 4 |
|---|---:|---:|---:|---:|
| In-degree ban đầu | 0 | 0 | 2 | 1 |

Khi học xong môn `u`, ta giảm 1 cho tất cả `v` trong `graph[u]`. Một môn **chỉ được mở khóa** khi `indegree` về 0.

In [4]:
indegree_demo = [0] * (n_demo + 1)
for u, v in rel_demo:
    indegree_demo[v] += 1
print('in-degree theo môn 1..4:', indegree_demo[1:])
print('Có thể học ngay:', [c for c in range(1, n_demo + 1) if indegree_demo[c] == 0])
assert indegree_demo[1:] == [0, 0, 2, 1]

in-degree theo môn 1..4: [0, 0, 2, 1]
Có thể học ngay: [1, 2]


## 6. Một lần cập nhật có ý nghĩa gì?

Sau học kỳ 1, đã xong môn 1 và 2. Hai cạnh `1→3` và `2→3` bị xóa khỏi phần đồ thị còn lại. Lúc đó `indegree[3]` từ 2 về 0, nên **kỳ 2** mới được học 3 (không phải ngay kỳ 1).

In [5]:
indegree_after = indegree_demo.copy()  # copy: không làm thay đổi dữ liệu mẫu
for completed in [1, 2]:
    for nxt in graph_demo[completed]:
        indegree_after[nxt] -= 1
print('Sau kỳ 1:', indegree_after[1:])
assert indegree_after[3] == 0 and indegree_after[4] == 1

Sau kỳ 1: [0, 0, 0, 1]


## 7. Cách dễ nghĩ: mô phỏng từng học kỳ và quét lại quan hệ

Giữ `taken`: tập môn **đã hoàn thành từ các kỳ trước**. Mỗi kỳ, quét các cạnh để tìm các môn còn bị chặn, chọn **đồng thời** tất cả các môn chưa học không bị chặn.

Cách này dễ suy luận nhưng mỗi học kỳ lại phải quét toàn bộ danh sách. Đây là *baseline* để kiểm chứng cách tối ưu.

In [6]:
def semesters_by_rescanning(n, relations):
    taken = set()
    semesters = 0
    while len(taken) < n:
        blocked = {v for u, v in relations if u not in taken}
        available = [v for v in range(1, n + 1)
                     if v not in taken and v not in blocked]
        if not available:
            return -1
        # Chỉ cập nhật taken SAU KHI đã tìm toàn bộ môn của kỳ này.
        taken.update(available)
        semesters += 1
    return semesters

assert semesters_by_rescanning(3, [[1,3],[2,3]]) == 2
assert semesters_by_rescanning(3, [[1,2],[2,3],[3,1]]) == -1
assert semesters_by_rescanning(4, rel_demo) == 3
print('Baseline: PASS')

Baseline: PASS


## 8. Trace baseline: lấy đúng một «ảnh chụp» cho mỗi học kỳ

Nếu vừa thêm môn 3 vào `taken` rồi lập tức cho môn 4 học **cùng kỳ**, thì sai luật. Phải tìm hết danh sách môn sẵn sàng trước, sau đó mới đánh dấu cả nhóm đã học xong.

In [7]:
def trace_rescanning(n, relations):
    taken = set()
    sem = 0
    while len(taken) < n:
        blocked = {v for u, v in relations if u not in taken}
        ready = [v for v in range(1, n+1) if v not in taken and v not in blocked]
        print(f'Kỳ {sem+1} | taken trước={sorted(taken)} | blocked={sorted(blocked)} | ready={ready}')
        if not ready:
            return -1
        taken.update(ready)
        sem += 1
    return sem
assert trace_rescanning(4, rel_demo) == 3

Kỳ 1 | taken trước=[] | blocked=[3, 4] | ready=[1, 2]
Kỳ 2 | taken trước=[1, 2] | blocked=[4] | ready=[3]
Kỳ 3 | taken trước=[1, 2, 3] | blocked=[] | ready=[4]


## 9. Nút thắt của baseline và ý tưởng cải tiến

Gọi `n` = số môn, `m` = số cạnh, `S` = số học kỳ (nếu không chu trình thì `S ≤ n`). Mỗi kỳ baseline duyệt tới `n` môn và `m` cạnh ⇒ **O(S(n+m))**, xấu nhất **O(n(n+m))**.

Ta không cần quét lại tất cả các cạnh. Khi môn `u` vừa hoàn thành, chỉ cần xử lý **các cạnh đi ra từ `u`**; mỗi cạnh vì thế được cập nhật đúng một lần.

## 10. Python cơ bản: hàng đợi `deque`

`deque` (double-ended queue) hỗ trợ `append(x)` ở cuối và `popleft()` ở đầu nhanh O(1). Dùng `list.pop(0)` có thể phải dịch chuyển các phần tử O(n).

Ở bài này, hàng đợi chứa các môn đã hết tiên quyết và sẵn sàng để học ở kỳ hiện tại hoặc kỳ tiếp theo.

In [8]:
from collections import deque
q_demo = deque([1, 2])
print('Ban đầu:', list(q_demo))
u = q_demo.popleft()
q_demo.append(3)
print('Lấy đầu:', u, '| Còn lại:', list(q_demo))
assert u == 1 and list(q_demo) == [2, 3]

Ban đầu: [1, 2]
Lấy đầu: 1 | Còn lại: [2, 3]


## 11. Bước đột phá: Topological Sort theo từng tầng

**Bước 1:** Xây `graph` và `indegree`.  
**Bước 2:** Bỏ vào `deque` tất cả môn có `indegree==0`.  
**Bước 3:** Mỗi lần xử lý **đúng `len(queue)` phần tử ở đầu kỳ**. Với từng môn `u`, giảm bậc vào của các môn phụ thuộc vào nó. Khi bậc vào của `v` bằng 0, thêm `v` vào queue (cho *kỳ sau*).  
**Bước 4:** Nếu học được `n` môn ⇒ số lần xử lý tầng là đáp án. Nếu không ⇒ có chu trình nên trả `-1`.

**Mấu chốt:** `size = len(q)` phải được chụp **trước** vòng `for`. Nếu cứ `while q` mà tăng học kỳ 1 lần cho tất cả môn thì sẽ trộn nhiều kỳ vào nhau.

In [9]:
def minimum_semesters_kahn(n, relations):
    graph = [[] for _ in range(n+1)]
    indegree = [0] * (n+1)
    for before, after in relations:
        graph[before].append(after)
        indegree[after] += 1

    q = deque(c for c in range(1, n+1) if indegree[c] == 0)
    completed = 0
    semesters = 0
    while q:
        size = len(q)                 # snapshot học kỳ này
        semesters += 1
        for _ in range(size):
            u = q.popleft()
            completed += 1
            for v in graph[u]:
                indegree[v] -= 1
                if indegree[v] == 0:
                    q.append(v)       # đợi học kỳ tiếp theo
    return semesters if completed == n else -1

assert minimum_semesters_kahn(3, [[1,3],[2,3]]) == 2
assert minimum_semesters_kahn(3, [[1,2],[2,3],[3,1]]) == -1
assert minimum_semesters_kahn(4, rel_demo) == 3
print('Kahn BFS: PASS')

Kahn BFS: PASS


## 12. Trace bằng tay (bảng trạng thái)

Input: `n=4`, `relations=[[1,3],[2,3],[3,4]]`.

| Đầu kỳ | Queue đầu kỳ | In-degree trước kỳ của 3,4 | Đã học xong sau kỳ | Queue mới |
|---|---|---|---|---|
| 1 | `[1,2]` | `(2,1)` | `1,2` | `[3]` |
| 2 | `[3]` | `(0,1)` | `1,2,3` | `[4]` |
| 3 | `[4]` | `(0,0)` | tất cả | `[]` |

Không thể học 4 sớm hơn kỳ 3 vì phải học xong chuỗi `1→3→4` hoặc `2→3→4` gồm ba môn liên tiếp.

In [10]:
def trace_kahn(n, relations):
    graph = [[] for _ in range(n+1)]
    indegree = [0]*(n+1)
    for u,v in relations:
        graph[u].append(v)
        indegree[v] += 1
    q = deque(v for v in range(1,n+1) if indegree[v]==0)
    completed = 0
    semester = 0
    while q:
        size = len(q)
        semester += 1
        print(f'\nKỳ {semester}: queue trước={list(q)}, indegree={indegree[1:]}')
        for _ in range(size):
            u = q.popleft()
            completed += 1
            print(f'  hoàn thành {u}')
            for v in graph[u]:
                before = indegree[v]
                indegree[v] -= 1
                print(f'    giảm indegree[{v}]: {before} -> {indegree[v]}')
                if indegree[v] == 0:
                    q.append(v)
        print('  queue cho kỳ sau:', list(q))
    result = semester if completed == n else -1
    print('Đã hoàn thành',completed,'/',n,'-> kết quả',result)
    return result

assert trace_kahn(4,rel_demo) == 3


Kỳ 1: queue trước=[1, 2], indegree=[0, 0, 2, 1]
  hoàn thành 1
    giảm indegree[3]: 2 -> 1
  hoàn thành 2
    giảm indegree[3]: 1 -> 0
  queue cho kỳ sau: [3]

Kỳ 2: queue trước=[3], indegree=[0, 0, 0, 1]
  hoàn thành 3
    giảm indegree[4]: 1 -> 0
  queue cho kỳ sau: [4]

Kỳ 3: queue trước=[4], indegree=[0, 0, 0, 0]
  hoàn thành 4
  queue cho kỳ sau: []
Đã hoàn thành 4 / 4 -> kết quả 3


## 13. Vì sao chu trình khiến bài toán vô nghiệm?

Nếu có chu trình `1→2→3→1` thì:
- muốn học 1 phải hoàn thành 3;
- muốn học 3 phải hoàn thành 2;
- muốn học 2 phải hoàn thành 1.

Không môn nào trong chu trình có thể là môn **đầu tiên**. Sau khi hàng đợi hết, vẫn còn môn chưa học: `completed < n` ⇒ **trả -1**. Một chu trình trong *một phần* đồ thị cũng dẫn đến kết quả `-1` dù các môn khác vẫn học được.

In [11]:
cycle_example = [[1, 2], [2, 3], [3, 1]]
print('Chu trình 3 môn:', minimum_semesters_kahn(3,cycle_example))
print('Chu trình trong đồ thị 5 môn:', minimum_semesters_kahn(5, [[1,2],[3,4],[4,3]]))
assert minimum_semesters_kahn(5, [[1,2],[3,4],[4,3]]) == -1

Chu trình 3 môn: -1
Chu trình trong đồ thị 5 môn: -1


## 14. Code nộp LeetCode (không in trace)

Dán nguyên class vào LeetCode. `List` trong chữ ký dùng theo skeleton; trong notebook import `typing.List` để chạy độc lập.

In [12]:
from typing import List
from collections import deque

class Solution:
    def minimumSemesters(self, n: int, relations: List[List[int]]) -> int:
        graph = [[] for _ in range(n+1)]
        indegree = [0]*(n+1)
        for u,v in relations:
            graph[u].append(v)
            indegree[v] += 1
        q = deque(v for v in range(1,n+1) if indegree[v]==0)
        semester = 0
        done = 0
        while q:
            semester += 1
            for _ in range(len(q)):
                u = q.popleft()
                done += 1
                for v in graph[u]:
                    indegree[v] -= 1
                    if indegree[v]==0:
                        q.append(v)
        return semester if done == n else -1

assert Solution().minimumSemesters(3,[[1,3],[2,3]]) == 2
assert Solution().minimumSemesters(3,[[1,2],[2,3],[3,1]]) == -1
print('Solution: PASS')

Solution: PASS


## 15. Chứng minh vì sao thuật toán trả *ít nhất* số học kỳ

**Bất biến:** đầu học kỳ `t`, queue chứa *đúng* các môn chưa học mà tất cả tiên quyết đã xong ở các học kỳ `< t`.

- **Cơ sở:** ở học kỳ 1, đúng những môn có in-degree 0 được học.
- **Duy trì:** sau kỳ `t`, ta xóa mỗi cạnh đi ra từ môn vừa học; môn mới đạt indegree 0 được đưa vào queue cho kỳ `t+1`. Không môn chưa đủ điều kiện nào bị thêm nhầm.
- **Tối ưu:** nếu môn đã đủ điều kiện thì cho nó học **ngay bây giờ** không làm cản trở bất kỳ môn nào khác (không có giới hạn số môn/kỳ). Trì hoãn chỉ có thể làm kết quả không tốt hơn. Vì vậy lịch lấy toàn bộ queue là tối ưu.

Cách nhìn khác: học kỳ của môn = `1 + max(học kỳ tiên quyết)` (nếu không có tiên quyết thì bằng 1). Đáp án là chiều dài đường đi dài nhất tính theo số **đỉnh** trên một đồ thị có hướng không chu trình (DAG).

## 16. Độ phức tạp

Gọi `n` = số môn, `m` = số quan hệ.

- Xây danh sách kề và bậc vào: **O(n+m)**.
- Mỗi môn vào/ra `deque` tối đa một lần, mỗi cạnh được duyệt đúng một lần: **O(n+m)** thời gian tổng.
- `graph`, `indegree`, `deque` chứa tối đa `n+m` phần tử: **O(n+m)** bộ nhớ.
- Baseline phải quét lại mỗi kỳ: **O(S(n+m))**, tệ nhất O(n(n+m)).

Đây là phân tích *worst-case*, không dựa vào việc queue may mắn ngắn.

## 17. Bộ test: mẫu đề, biên và tình huống dễ nhầm

Trường hợp `relations=[]` dưới đây là **mở rộng tự nhiên** (không nằm trong giới hạn `len(relations) >= 1` của bản mô tả tham khảo). Nó dùng để kiểm tra tính tổng quát của thuật toán.

In [13]:
cases = [
    ('ví dụ 1',3,[[1,3],[2,3]],2),
    ('ví dụ 2 - chu trình',3,[[1,2],[2,3],[3,1]],-1),
    ('không tiên quyết (mở rộng)',1,[],1),
    ('nhiều môn độc lập (mở rộng)',5,[],1),
    ('chuỗi',4,[[1,2],[2,3],[3,4]],4),
    ('hình thoi',4,[[1,2],[1,3],[2,4],[3,4]],3),
    ('chu trình ở một phần',5,[[1,2],[3,4],[4,3]],-1),
]
for name,n,relations,expected in cases:
    actual = Solution().minimumSemesters(n,relations)
    assert actual == expected,(name,actual,expected)
    print(f'{name}: PASS ({actual})')

ví dụ 1: PASS (2)
ví dụ 2 - chu trình: PASS (-1)
không tiên quyết (mở rộng): PASS (1)
nhiều môn độc lập (mở rộng): PASS (1)
chuỗi: PASS (4)
hình thoi: PASS (3)
chu trình ở một phần: PASS (-1)


## 18. Test đối chiếu ngẫu nhiên với baseline độc lập

Tạo quan hệ ngẫu nhiên **không trùng** và không tự trỏ; kiểm tra Kahn và baseline cho cùng kết quả. Những ca nhỏ này đặc biệt hữu ích để phát hiện việc gộp nhầm hai học kỳ.

In [14]:
import random
random.seed(1136)
for case_no in range(250):
    n = random.randint(1, 8)
    pairs = [(u,v) for u in range(1,n+1) for v in range(1,n+1) if u != v]
    random.shuffle(pairs)
    relations = [list(edge) for edge in pairs[:random.randint(0,min(15,len(pairs)))]]
    slow = semesters_by_rescanning(n,relations)
    fast = Solution().minimumSemesters(n,relations)
    assert fast == slow,(case_no,n,relations,slow,fast)
print('250 đồ thị ngẫu nhiên: PASS')

250 đồ thị ngẫu nhiên: PASS


## 19. Kiểm thử ở quy mô lớn và tránh lỗi đệ quy

`n = 5000` đủ lớn để DFS đệ quy sâu theo chuỗi có thể vượt giới hạn recursion mặc định của Python. **Kahn + deque là dạng lặp**, không gặp rủi ro này.

In [15]:
chain = [[i, i+1] for i in range(1, 5000)]
assert Solution().minimumSemesters(5000,chain) == 5000
assert Solution().minimumSemesters(5000,chain+[[5000,1]]) == -1
print('5000 môn: chuỗi dài và chu trình dài PASS')

5000 môn: chuỗi dài và chu trình dài PASS


## 20. Những lỗi người mới hay gặp

1. **Lấy sai chiều cạnh:** `graph[v].append(u)` thay vì `graph[u].append(v)` sẽ hiểu ngược tiên quyết.
2. **Cho môn được mở khóa học ngay cùng kỳ:** chạy `while q` bên trong một học kỳ mà không chụp `len(q)`.
3. **Không đếm số môn hoàn tất:** khi có chu trình mà queue rỗng, lại trả số học kỳ đã đi qua.
4. **Nhầm môn 1-based với index 0-based:** `range(1,n+1)` chứ không phải `range(n)`.
5. **Dùng `[[]] * (n+1)`** làm các danh sách kề dùng chung một object.

In [16]:
def wrong_mix_semesters(n, relations):
    """Sai: học tất cả môn có thể đến trong 1 lần while => số kỳ thường là 1."""
    graph = [[] for _ in range(n+1)]
    indeg = [0]*(n+1)
    for u,v in relations:
        graph[u].append(v); indeg[v] += 1
    q = deque(v for v in range(1,n+1) if indeg[v]==0)
    done = 0
    while q:
        u = q.popleft(); done += 1
        for v in graph[u]:
            indeg[v]-=1
            if indeg[v]==0: q.append(v)
    return 1 if done==n else -1

print('Sai (chuỗi 3 môn):',wrong_mix_semesters(3,[[1,2],[2,3]]))
print('Đúng:',minimum_semesters_kahn(3,[[1,2],[2,3]]))
assert wrong_mix_semesters(3,[[1,2],[2,3]]) == 1
assert minimum_semesters_kahn(3,[[1,2],[2,3]]) == 3

Sai (chuỗi 3 môn): 1
Đúng: 3


## 21. Biến thể: học kỳ sớm nhất của *từng* môn

Đôi khi đề không chỉ hỏi tổng số kỳ, mà hỏi môn `v` có thể được học sớm nhất vào kỳ mấy. Công thức:

`term[v] = max(term[v], term[u] + 1)` cho mỗi cạnh `u→v`, với các môn không có tiên quyết bắt đầu ở kỳ `1`.

Ta vẫn tính trên thứ tự topo. Với chu trình, không tồn tại lịch đầy đủ.

In [17]:
def earliest_terms(n,relations):
    graph = [[] for _ in range(n+1)]
    indeg = [0]*(n+1)
    for u,v in relations:
        graph[u].append(v); indeg[v] += 1
    q=deque(x for x in range(1,n+1) if indeg[x]==0)
    terms = [1]*(n+1)
    done=0
    while q:
        u=q.popleft();done+=1
        for v in graph[u]:
            terms[v] = max(terms[v], terms[u]+1)
            indeg[v]-=1
            if indeg[v]==0:q.append(v)
    return terms[1:] if done==n else None

assert earliest_terms(4,rel_demo) == [1,1,2,3]
assert earliest_terms(3,cycle_example) is None
print('Kỳ sớm nhất mỗi môn:',earliest_terms(4,rel_demo))

Kỳ sớm nhất mỗi môn: [1, 1, 2, 3]


## 22. Bài tập vận dụng — hãy tự làm trước khi đọc đáp án

**Bài 1 — Tính tay.** `n=5`, quan hệ `[[1,3],[2,3],[3,4],[2,5]]`. Hãy viết các học kỳ và kết quả.

**Bài 2 — Đoán output.** Với `n=4`, `[[1,2],[1,3],[2,4],[3,4]]`, bậc vào ban đầu của môn 1..4 là gì?

**Bài 3 — TODO.** Viết `exercise_indegree(n,relations)` trả danh sách bậc vào của môn 1..n.

**Bài 4 — TODO.** Viết `exercise_min_semesters(n,relations)` theo Kahn BFS (không gọi `Solution`).

**Bài 5 — Mini challenge.** Viết `exercise_earliest_terms(n,relations)` trả kỳ sớm nhất của mỗi môn hoặc `None` nếu có chu trình.

Các hàm TODO dưới đây **chưa được gọi**, nên người học có thể mở notebook và Run All an toàn.

In [18]:
def exercise_indegree(n,relations):
    # TODO: tạo mảng có n+1 phần tử; tăng indeg[v] với từng cạnh u->v
    pass

def exercise_min_semesters(n,relations):
    # TODO: graph, indegree, deque, xử lý từng kỳ theo len(queue), phát hiện chu trình
    pass

def exercise_earliest_terms(n,relations):
    # TODO: kết hợp Kahn với term[v] = max(term[v], term[u]+1)
    pass

# Khi làm xong, bỏ comment để tự test:
# assert exercise_indegree(4,rel_demo) == [0,0,2,1]
# assert exercise_min_semesters(4,rel_demo) == 3
# assert exercise_earliest_terms(4,rel_demo) == [1,1,2,3]

## 23. Gợi ý 3 mức (chỉ mở nếu đang bí)

| Bài | Gợi ý tư duy | Gợi ý cú pháp | Gần-code |
|---|---|---|---|
| 1 | Tìm môn đầu vào bằng 0 | Học 1,2 trước | Sau đó 3 và 5, rồi 4 |
| 2 | Đếm mũi tên đi *vào* | `indeg[v] += 1` | Kiểm tra 4 nhận từ 2 và 3 |
| 3 | Chỉ cần duyệt các cạnh | `[0]*(n+1)` | Trả `indeg[1:]` |
| 4 | Chụp số môn đầu kỳ | `for _ in range(len(q))` | `completed==n` mới được trả số kỳ |
| 5 | Đường dài nhất theo topo | `term[v]=max(...)` | Chỉ đưa `v` vào queue khi `indeg[v]==0` |

## 24. Đáp án tham khảo — hãy tự đối chiếu sau khi làm

**Bài 1:** kỳ 1: `[1,2]`; kỳ 2: `[3,5]`; kỳ 3: `[4]`. Kết quả **3 kỳ**.

**Bài 2:** bậc vào môn 1..4 là `[0,1,1,2]`.

Ba hàm sau là **đáp án độc lập**, không thay đổi TODO của người học ở trên.

In [19]:
def answer_indegree(n,relations):
    indeg=[0]*(n+1)
    for u,v in relations: indeg[v]+=1
    return indeg[1:]

def answer_min_semesters(n,relations):
    graph=[[] for _ in range(n+1)]
    indeg=[0]*(n+1)
    for u,v in relations:
        graph[u].append(v); indeg[v]+=1
    q=deque(u for u in range(1,n+1) if indeg[u]==0)
    done=sem=0
    while q:
        sem+=1
        for _ in range(len(q)):
            u=q.popleft();done+=1
            for v in graph[u]:
                indeg[v]-=1
                if indeg[v]==0:q.append(v)
    return sem if done==n else -1

def answer_earliest_terms(n,relations):
    return earliest_terms(n,relations)

assert answer_indegree(4,rel_demo)==[0,0,2,1]
assert answer_min_semesters(4,rel_demo)==3
assert answer_earliest_terms(4,rel_demo)==[1,1,2,3]
print('Đáp án bài tập: PASS')

Đáp án bài tập: PASS


## 25. Checklist trước khi submit

```text
u -> v (học u trước v)
indegree[v] += 1
q = tất cả môn có indegree 0
Mỗi học kỳ: chỉ xử lý số phần tử q ban đầu
Xử lý u -> giảm indegree của các v phụ thuộc
indegree[v] == 0 -> đưa v vào q cho kỳ sau
Hoàn thành n môn? số kỳ; còn môn? -1
```

- [ ] Đúng chữ ký `minimumSemesters` và ID môn đánh số **1..n**.
- [ ] `len(q)` được lấy ở **đầu** mỗi kỳ.
- [ ] Không học một môn cùng kỳ với tiên quyết vừa hoàn thành.
- [ ] Có test chu trình, nhánh song song và chuỗi dài.
- [ ] Hiểu vì sao tham lam lấy tất cả môn sẵn sàng là tối ưu.